#Fine-Tuning (Adapting the Expert)

**Logic:** In Days 71 and 72, we "froze" the entire pre-trained model. It was like hiring an expert but telling them, "Don't change your thinking at all." Fine-Tuning means unfreezing the last few layers of the pre-trained model. This allows the model to adapt its high-level feature detection (like specific shapes) to your specific dataset.

This is a two-step process. First, you train the head (Day 71/72 logic), then you "unfreeze" and train at a very low learning rate.

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, optimizers

# 1. Load ResNet50 Base
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Step 1: Freeze everything and train the custom head first
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# 3. Step 2: UNFREEZE the last few layers for Fine-Tuning
# We unfreeze the last 10 layers of ResNet
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False

# 4. CRITICAL: Use a very small learning rate for fine-tuning
# We don't want to "destroy" the pre-trained weights, just nudge them.
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), # 0.00001
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,112,513 (91.98 MB)

 Trainable params: 4,990,465 (19.04 MB)

 Non-trainable params: 19,122,048 (72.94 MB)

#⚠️ The Golden Rules of Fine-Tuning

1. **Train the Head First:** Never fine-tune a model with a "randomly initialized" head. The huge errors from the new head will destroy the pre-trained weights in the base.

2. **Low Learning Rate:** Always use a learning rate 10x or 100x smaller than usual (e.g., 1e-5).

3. **Unfreeze Sparingly:** Only unfreeze the top layers (the ones closest to the output). The bottom layers (the ones closest to the input) find basic edges and should stay frozen.